# 🎯 Stratified Shuffle Split

When building a Machine Learning model, one of the most important steps is **splitting the dataset** into:

- **Training Set**
- **Testing Set**

This ensures that the model is evaluated on data it has **never seen before**, giving a realistic measure of its performance.

---

# 📂 Why Do We Split the Dataset?

If we train and test the model using the same data, the model may simply memorize the data instead of learning patterns.

This can lead to **overly optimistic results**.

To avoid this, we split the dataset into two parts:

- **Training Set** → Used to train the model.
- **Testing Set** → Used to evaluate the model.

A common split is:

```text
Training Set → 80%

Testing Set → 20%
```

---

# ⚠️ The Problem of Data Snooping Bias

One major issue during model development is **Data Snooping Bias**.

## What is Data Snooping Bias?

Data Snooping Bias occurs when information from the **test set leaks into the training process**.

As a result:

- The model appears to perform better than it actually does.
- It performs poorly on new, unseen data.

---

## Example

Suppose you have 1,000 house records.

If you look at all the data before creating the test set, you may unknowingly use information from the test data while preparing the model.

This causes the model to "cheat."

Therefore:

> **Always separate the test set before exploring or training the data.**

---

# 🎲 Random Sampling

The simplest way to split data is **Random Sampling**.

Steps:

1. Shuffle the dataset.
2. Select a percentage for testing.
3. Use the remaining data for training.

---

## Example

```python
import numpy as np

def shuffle_and_split_data(data, test_ratio):
    np.random.seed(42)

    shuffled_indices = np.random.permutation(len(data))

    test_set_size = int(len(data) * test_ratio)

    test_indices = shuffled_indices[:test_set_size]

    train_indices = shuffled_indices[test_set_size:]

    return data.iloc[train_indices], data.iloc[test_indices]
```

---

# 🔑 Why Use `np.random.seed(42)`?

Every time random sampling is performed, different rows may be selected.

Setting a **random seed** ensures that the same random split is generated every time the code runs.

This makes experiments:

- Reproducible
- Easier to debug
- Easier to compare

Example:

```python
np.random.seed(42)
```

The number **42** is simply a commonly used fixed value; any integer can be used.

---

# ⚠️ Problem with Random Sampling

Random sampling works well for many datasets.

However, it may accidentally produce an unbalanced train-test split.

### Example

Suppose:

| Income Group | Dataset |
|--------------|---------|
| Low Income | 60% |
| Medium Income | 30% |
| High Income | 10% |

A random split might create:

| Income Group | Test Set |
|--------------|----------|
| Low Income | 75% |
| Medium Income | 20% |
| High Income | 5% |

The distribution has changed.

This may reduce the reliability of model evaluation.

---

# 📊 Stratified Sampling

To solve this problem, we use **Stratified Sampling**.

Instead of selecting random records, the dataset is divided into groups called **strata**.

The train and test sets are then created so that each group has the **same proportion** as in the original dataset.

---

# 🧠 What is a Strata?

A **strata** is a subgroup of the dataset based on a particular feature.

Examples:

- Income Category
- Gender
- Age Group
- Education Level

Each subgroup is represented proportionally in both the training and testing sets.

---

# 🏠 Example: California Housing Dataset

One of the most important features affecting house prices is:

**Median Income**

Instead of randomly splitting the dataset, we first divide median income into categories.

These categories become our strata.

---

# 📋 Creating Income Categories

```python
import pandas as pd
import numpy as np

data = pd.read_csv("housing.csv")

data["income_cat"] = pd.cut(
    data["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)
```

---

# 📖 Understanding the Code

### `pd.cut()`

Used to divide continuous values into categories.

### Bins

```text
0 – 1.5     → Category 1

1.5 – 3.0   → Category 2

3.0 – 4.5   → Category 3

4.5 – 6.0   → Category 4

6.0+        → Category 5
```

A new column called **income_cat** is created.

---

# 📊 Visualizing the Income Categories

```python
import matplotlib.pyplot as plt

data["income_cat"].value_counts().sort_index().plot.bar(
    rot=0,
    grid=True
)

plt.title("Income Categories Distribution")
plt.xlabel("Income Category")
plt.ylabel("Number of Houses")

plt.show()
```

The bar chart shows the number of houses in each income category.

---

# 🔀 Stratified Shuffle Split

Scikit-Learn provides a built-in class called:

```text
StratifiedShuffleSplit
```

It automatically performs stratified sampling.

---

# 🐍 Code Example

```python
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_index, test_index in split.split(
    data,
    data["income_cat"]
):
    strat_train_set = data.loc[train_index]
    strat_test_set = data.loc[test_index]
```

---

# 📖 Understanding the Parameters

### `n_splits=1`

Perform only one train-test split.

---

### `test_size=0.2`

20% of the data goes to the test set.

80% remains for training.

---

### `random_state=42`

Ensures the split is reproducible.

---

### `split(data, data["income_cat"])`

Creates the split while maintaining the same proportion of each income category in both datasets.

---

# 📊 Random Sampling vs Stratified Sampling

## Random Sampling

```text
Dataset

🔵🔵🔵🟢🟢🔴

↓

Train

🔵🟢🔴

Test

🔵🔵🟢
```

Some groups may become overrepresented or underrepresented.

---

## Stratified Sampling

```text
Dataset

🔵🔵🔵🟢🟢🔴

↓

Train

🔵🔵🟢🔴

Test

🔵🟢
```

Every group keeps approximately the same proportion.

---

# ✅ Advantages of Stratified Sampling

- Better representation of data
- Reduces sampling bias
- Produces more reliable evaluation
- Especially useful for imbalanced datasets

---

# 📋 Summary

- Splitting data into training and testing sets helps evaluate model performance on unseen data.
- **Data Snooping Bias** occurs when information from the test set leaks into the training process.
- **Random Sampling** randomly divides the dataset but may create an unbalanced distribution.
- **Stratified Sampling** preserves the proportion of important groups (strata) in both the training and testing sets.
- **StratifiedShuffleSplit** in Scikit-Learn automates this process and is commonly used for reliable model evaluation.

---

# 🧠 Quick Exam Notes

### Data Snooping Bias

> Leakage of test-set information into the training process, leading to overly optimistic results.

### Random Sampling

> Randomly splits the dataset into training and testing sets.

### Stratified Sampling

> Splits the dataset while preserving the proportion of important groups (strata).

### Strata

> A subgroup of data based on a specific attribute.

### Why Create `income_cat`?

> To preserve the distribution of median income in both the training and testing datasets.

### Scikit-Learn Class

```python
from sklearn.model_selection import StratifiedShuffleSplit
```

### Common Split

```text
Training Set → 80%

Testing Set → 20%
```